# 13 · Right checks and the Right job

This notebook runs the Right half of the protocol (`dtrack_right`) against your real Databricks data. It covers the row check (`reconcile_partitions`, `evaluate_gate`, `reconcile_pair`, `run_row_check`), the column check (`compare_statistics`, `DrainScheduler`, `ColCheck`) and the whole job loop (`dtrack_right.job.run`). It reads and writes the same S3 run folder layout as the Left side. Pass the run printed by notebook 10 (`DTRACK_NB_RUN`), or let this notebook seed a fresh one.

Mirrors `tests/unit/dtrack_right/checks/test_row_check.py`, `tests/unit/dtrack_right/checks/test_col_check.py` and `tests/unit/dtrack_right/test_job.py`. Those tests use a memory store, a fake target and synthetic CSVs. Submitting the real Databricks job is gated by `DRY_RUN`.

## How to approach

**Can `dtrack_right` run on this machine?** Yes. In production it runs inside Databricks (`src/dtrack_right/notebook.py` with `SparkTarget(spark, ...)`), but the code only needs a *target* object with `preflight`, `partition_counts` and `window_statistics`. It also needs a store, and `S3Store` / `VolumeStore` work from anywhere. Two real targets work locally:
- **`TARGET_MODE = "warehouse"` (default, main `.venv`)**: `dtrack_local.pipeline.WarehouseTarget` subclasses `SparkTarget` and sends the same SQL through the Statement Execution API of a SQL warehouse. Counts and `DESCRIBE QUERY` run in the warehouse. For statistics, it pulls the window's raw rows to a local CSV and computes them with the same pandas code as the Left side. This is the path the prod smoke plan uses.
- **`TARGET_MODE = "connect"`**: the production `SparkTarget` with a Databricks Connect session, so statistics are computed in Spark SQL exactly as in the job. This needs the separate `.venv-live` kernel (`just live sync`: Python 3.12 + `databricks-connect`) and serverless or a cluster (`DBX_COMPUTE`).

`python -m dtrack_right` is not a way in: it only wires the local `CsvAnchorTarget` (CSV files) used by the tests.

**Prerequisites.** Kernel `.venv` (`uv sync --extra dbx`) or `.venv-live` for connect mode. You need the Left credentials in `.env` (only when this notebook seeds the Left side itself), AWS credentials that can write `S3_SCRATCH`, a Databricks profile and a SQL warehouse id.

**Run order.** Run it top to bottom.
- Sections 1-4 do the row check: Right counts are a GROUP BY in the warehouse.
- Section 5 makes sure the Left units exist (runs the Left column check if notebook 10 did not).
- Sections 6-8 do the column check. Each window is one warehouse pull in warehouse mode.
- Section 9 runs `job.run`, the job's main loop, locally against the same run.
- Section 10 submits the real Databricks notebook job, only with `DRY_RUN = False`.

**What to look at.** Check the `col_map` spelling against `DESCRIBE`, the per-day verdicts and the gate decision, the per-column comparison table, and the final `summary.json` (`status`, `findings`).

**Failure modes**
- `Right-side preflight failed`: the table cannot be found, or `date_col` is spelled differently.
- `KeyError` on a column in section 6: a `col_map` right name differs in case from `DESCRIBE`. This is a known open bug, and section 2 flags it.
- `TimeoutError: right idle timeout`: the Left side never closed its units.
- `left heartbeat is stale`: the Left process died mid-run.
- `ImportError: databricks.connect`: you are in the wrong kernel for connect mode.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# The project .env first, so the DTRACK_* defaults below (and Left credentials, if seeding) resolve
ENV_FILE = os.environ.get("DTRACK_ENV_FILE", "C:/work/dtrack/.env")   # repo .env on this machine
load_dotenv(ENV_FILE, override=False)

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")   # your real pairs config
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")             # one active pair
TO_DATE = None                                                 # None = yesterday (only when seeding here)
DAYS = 3                                                       # window length (only when seeding here)
S3_SCRATCH = os.environ.get("DTRACK_PROD_S3", "s3://<bucket>/dtrack/nb-scratch")   # the only S3 writes go here
LEFT_RUN = os.environ.get("DTRACK_NB_RUN", "")                # run URI from notebook 10 (KEEP=True); "" = seed a run here
DBX_PROFILE = os.environ.get("DTRACK_DBX_PROFILE", "corp")    # ~/.databrickscfg profile
WAREHOUSE_ID = os.environ.get("DTRACK_DBX_WAREHOUSE_ID", "")  # SQL warehouse id (warehouse mode)
DISPOSITION = os.environ.get("DTRACK_LOCAL_DISPOSITION", "external_links")   # inline when presigned downloads are blocked
TARGET_MODE = "warehouse"                                      # "warehouse" (main .venv) | "connect" (.venv-live, Databricks Connect)
DBX_COMPUTE = os.environ.get("DTRACK_LIVE_COMPUTE", "serverless")   # connect mode: "serverless" or a cluster id
GATE = "auto"                                                  # auto | manual | ratio:<0..1>
MAX_ROWS = 200_000                                             # refuse a column check bigger than this many Left rows
IDLE_TIMEOUT = 300                                             # seconds the drain waits without progress
WORKSPACE_DIR = os.environ.get("DTRACK_NB_WORKSPACE_DIR", "") # Workspace folder for the job upload; "" = /Users/<me>/dtrack-nb13
AWS_SECRET_SCOPE = os.environ.get("DTRACK_LIVE_AWS_SECRET_SCOPE", "")   # secret scope with the AWS key, for an s3:// work location
LOG_LEVEL = "INFO"                                             # DEBUG shows every SQL statement
WORKDIR = Path.home() / ".local/plans/nbs/_work/13"           # Right pulls, Left pulls, report files
KEEP = False                                                   # True = keep the scratch objects this notebook wrote
DRY_RUN = True                                                 # False = upload dtrack_right and submit a real Databricks job run

In [ ]:
import copy
import json
from datetime import date, timedelta

import pandas as pd

from dtrack_left.checks.row_check import make_run_id
from dtrack_left.cli.config import load_config
from dtrack_left.cli.options import setup_logging

# dtrack logs (UTC timestamps) go to stderr, which Jupyter shows under each cell
setup_logging(LOG_LEVEL)

# Load and validate the real config: conn_macro profiles merged, col_map files expanded
config = load_config(CONFIG_PATH)
active = [name for name, row in config["pairs"].items() if not row.get("skip")]
print(f"{len(active)} active pair(s): {active}")
if PAIR not in active:
    raise ValueError(f"PAIR={PAIR!r} is not an active pair of {CONFIG_PATH}")

# The check window: the last DAYS days up to TO_DATE (default yesterday), both ends inclusive
TO = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
FROM = (date.fromisoformat(TO) - timedelta(days=DAYS - 1)).isoformat()

# A deep copy of the pair with its dates narrowed to the window; the real config is never changed
pair = copy.deepcopy(config["pairs"][PAIR])
pair["fromDate"], pair["toDate"] = FROM, TO
run_config = {"pairs": {PAIR: pair}}
DATE_COL = pair["left"]["date_col"]
KEYS = [k.strip() for k in str(pair.get("key") or "").split(",") if k.strip()]
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"pair {PAIR}: {pair['left'].get('source')} -> databricks, window {FROM}..{TO}, key={KEYS or None}")
print(f"col_map: {len(pair['col_map'])} column(s), sample={pair.get('sample')!r}, vintage={pair.get('vintage') or 'day'}")

from dtrack_left.checks.row_check import run_row_check as left_row_check
from dtrack_right.backends.store import open_store, resolve_run, run_is_complete

# A run from notebook 10, or a fresh one under this notebook's own scratch root
if LEFT_RUN:
    RUN_URI = LEFT_RUN.rstrip("/")
    RUN_ID = RUN_URI.rsplit("/", 1)[-1]
    NB_ROOT = RUN_URI.rsplit("/runs/", 1)[0]
else:
    RUN_ID = make_run_id(run_config)
    NB_ROOT = f"{S3_SCRATCH.rstrip('/')}/nb13-{RUN_ID}"
    RUN_URI = f"{NB_ROOT}/runs/{RUN_ID}"
REPORT_URI = f"{NB_ROOT}/report"
store = open_store(RUN_URI)

# Seed the Left half only when the run has no Left counts for the pair yet
if not store.exists(f"row_check/{PAIR}.left.json"):
    print("seeding the Left row check here:", RUN_URI)
    left_row_check(run_config, store, run_id=RUN_ID, pair_names=[PAIR])
left_doc = store.get_json(f"row_check/{PAIR}.left.json")
right_spec = left_doc["right"]
print(f"RUN_URI = {RUN_URI}\nwindow  = {left_doc['bounds']}  ({len(left_doc['counts'])} Left day(s))")

## 1. The run folder: stores and run resolution

`open_store` picks `S3Store` for `s3://` and `VolumeStore` for a Volume or local path. Both have the same contract. `resolve_run` is how the job finds its run. An explicit `run_id` wins. A location that already is a run is used as-is. Otherwise it takes the newest run under a runs root that has no `_STATUS.json` with `state: complete`.

In [ ]:
from dtrack_right.backends.store import VolumeStore

# How the job would resolve: the run itself, and the runs root above it (newest unfinished run)
runs_root = RUN_URI.rsplit("/", 1)[0]
for location, run_id in ((RUN_URI, None), (runs_root, None), (runs_root, RUN_ID)):
    resolved, name = resolve_run(location, run_id)
    print(f"resolve_run({location!r}, run_id={run_id!r}) -> {name} ({type(resolved).__name__ if resolved else None})")
print("run complete?   :", run_is_complete(store))
print("runs under root :", open_store(runs_root).list_prefixes())
print("row_check/      :", store.list_keys("row_check/"))
print("_STATUS.json    :", store.get_json("_STATUS.json") if store.exists("_STATUS.json") else None)

# The same contract on a local folder (what a Volume path gives the job inside Databricks)
local = VolumeStore(WORKDIR / "volume_demo")
local.put_json("probe.json", {"ok": True})
print("VolumeStore     :", local.root_uri, local.list_keys(""), local.get_json("probe.json"))

## 2. The Right target and its preflight

`preflight` runs `DESCRIBE QUERY SELECT * FROM <relation>` and checks `date_col`. `relation()` shows what the target reads for the pair's `kind` (table, query or files). The `col_map` check below is the prod smoke test 4. Databricks names are case-insensitive in SQL, but the Right statistics look up column types by exact name. So a right name with a different case silently turns a numeric column into a categorical one.

In [ ]:
from dtrack_right.backends.targets import SparkTarget

# Build the Right target factory: SQL warehouse (default) or Databricks Connect (production SparkTarget)
if TARGET_MODE == "connect":
    from databricks.connect import DatabricksSession

    os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", DBX_PROFILE)
    builder = DatabricksSession.builder
    spark = (builder.serverless() if DBX_COMPUTE == "serverless" else builder.clusterId(DBX_COMPUTE)).getOrCreate()
    target_factory = lambda right: SparkTarget(spark, right)
else:
    from dtrack_left.backends.databricks import workspace
    from dtrack_local.pipeline import WarehouseTarget
    from dtrack_local.warehouse import Warehouse

    warehouse = Warehouse(workspace(DBX_PROFILE), WAREHOUSE_ID, disposition=DISPOSITION)
    target_factory = lambda right: WarehouseTarget(warehouse, right, WORKDIR / "right")

target = target_factory(right_spec)
health = target.preflight()
print(type(target).__name__, "| relation:", target.relation())
print("preflight ok:", health["ok"], "|", health.get("error") or f"{len(health['columns'])} column(s)")

# col_map right names must match DESCRIBE exactly (case included)
actual = set(health.get("columns") or [])
wanted = [right_spec["date_col"], *pair["col_map"].values()]
wrong = {w: next((a for a in actual if a.lower() == w.lower()), "missing") for w in wanted if w not in actual}
print("col_map right names to fix:", wrong or "none")
pd.Series(getattr(target, "column_types", {}), name="type").to_frame()

## 3. Right counts, reconciliation and the gate

`partition_counts` is one `GROUP BY` day over the window bounds. `reconcile_partitions` classifies every day on either side as `match`, `count_diff` (with `delta` = right − left), `left_only` or `right_only`, after folding `date_synonyms` on both sides. `evaluate_gate` decides whether Stage 2 may start. `auto` needs at least one match, `manual` never passes, and `ratio:x` needs the excluded share ≤ x. These are pure functions, so all the policies are evaluated on your real days.

In [ ]:
from dtrack_right.checks.row_check import evaluate_gate, reconcile_partitions, summarize_partitions
from dtrack_right.common.dates import apply_synonyms

# Count the Right side for exactly the bounds the Left side counted
bounds = left_doc["bounds"]
right_counts = target.partition_counts(bounds.get("from"), bounds.get("to"))
synonyms = left_doc.get("date_synonyms") or {}
print(f"right: {len(right_counts)} day(s), {sum(right_counts.values())} row(s); date_synonyms={synonyms or 'none'}")
if synonyms:
    print("left after synonyms :", apply_synonyms(left_doc["counts"], synonyms))

# Per-day verdicts and their summary
partitions = reconcile_partitions(left_doc["counts"], right_counts, synonyms)
print("summary:", summarize_partitions(partitions))

# Every gate policy on the same real days
for policy in ("auto", "manual", "ratio:0", "ratio:0.25", "ratio:1"):
    decision = evaluate_gate(partitions, policy)
    print(f"{policy:11} passed={decision['passed']!s:5} excluded_ratio={decision['excluded_ratio']:.3f} matched={decision['matched_partitions']}/{decision['total_partitions']}")
try:
    evaluate_gate(partitions, "ratio:1.5")
except ValueError as exc:
    print("ratio:1.5   ValueError:", exc)
pd.DataFrame(partitions).T

## 4. The Right row check on the run

`run_row_check` reconciles every pending `row_check/<pair>.left.json` (`pending_left_documents`) through `reconcile_pair`. It writes `reports/<pair>_compare_row.csv` first, then `manifest/<pair>.json` if the gate passed, and the verdict `row_check/<pair>.json` last. The verdict is the commit marker, so a crash before it leaves the pair pending and it is redone on restart. Once the Left side has closed and every announced pair has a verdict, it writes `row_check/_CLOSED`. A second call finds nothing pending.

In [ ]:
from dtrack_right.checks.row_check import pending_left_documents, row_check_csv_bytes, run_row_check

# Reconcile what is pending (0 if notebook 10 already played the Right side), then call again: nothing pending
print("pending before:", [doc["pair"] for doc in pending_left_documents(store)])
print("reconciled    :", run_row_check(store, target_factory, gate=GATE))
print("again         :", run_row_check(store, target_factory, gate=GATE))
print("row_check/_CLOSED:", store.get_json("row_check/_CLOSED") if store.exists("row_check/_CLOSED") else "not yet")

# The verdict document, its CSV, and the released manifest (if the gate passed)
verdict = store.get_json(f"row_check/{PAIR}.json")
print("gate:", verdict["gate_decision"], "| timing:", verdict["timing"])
print(row_check_csv_bytes(verdict).decode()[:1500])
manifest = store.get_json(f"manifest/{PAIR}.json") if store.exists(f"manifest/{PAIR}.json") else None
if manifest:
    print(f"manifest: {len(manifest['windows'])} window(s) x {len(manifest['columns'])} column(s) = {manifest['expected_units']} unit(s); sample={manifest['sample']}")
else:
    print("no manifest: the gate did not pass, so Stage 2 is skipped for this pair")

## 5. Make sure the Left units exist

The Right column check compares Left unit payloads (`units/`, pointed to by `_INDEX/`) with its own statistics. If notebook 10 already ran its column check on this run, they are there. Otherwise this cell runs the Left column check on this machine (`LeftRuntime` + `ColCheckPipeline`, the body of `dtrack col-check`), which pulls from your Left source.

In [ ]:
from dtrack_left.checks.col_check import ColCheckPipeline, LeftRuntime, publish_fixed_keys, wait_for_manifests

if manifest is None:
    raise RuntimeError("no manifest released for this pair; Stage 2 cannot run")
indexed = [row for _key, row in store.list_json("_INDEX/") if row["pair"] == PAIR]
if indexed:
    print(f"{len(indexed)} Left unit(s) already published (notebook 10)")
else:
    left_rows = sum(w["n_left"] for w in manifest["windows"])
    if left_rows > MAX_ROWS:
        raise RuntimeError(f"{left_rows} Left rows > MAX_ROWS={MAX_ROWS}: lower DAYS or raise MAX_ROWS")
    manifests = wait_for_manifests(store, timeout=0, poll_interval=0, pair_names=[PAIR])
    for item in manifests:
        publish_fixed_keys(store, item, WORKDIR / "left_keys")
    runtime = LeftRuntime(WORKDIR / "left", store)
    print("Left units published:", ColCheckPipeline(store, pull_window=runtime.pull_window, compute_stats=runtime.compute_stats).run(manifests))
print("manifest/_CLOSED:", store.get_json("manifest/_CLOSED") if store.exists("manifest/_CLOSED") else "missing")

## 6. One window by hand: Right statistics vs Left units

`window_statistics` gives the Right side's per-column statistics for one window. In warehouse mode, `window_sql` shows the exact pull. `compare_statistics` turns a Left/Right pair of statistics into one result row. Counts (`n_total`, `n_missing`, `n_unique`) must be equal. `mean`/`std`/lengths/`hash_sum` are compared with a float tolerance. Extremes are compared by value for numeric columns and verbatim for text. For fixed-N samples the window first gets the shared `key_tuples`, as `ColCheck.poll` does.

In [ ]:
from dtrack_right.checks.col_check import compare_statistics

# The first window, with the fixed-N keys attached when the sample is fixed
window = copy.deepcopy(manifest["windows"][0])
if manifest["sample"]["mode"] == "fixed":
    window["key_tuples"] = store.get_json(f"manifest/{PAIR}.keys.json")["windows"].get(window["bucket"], [])
if hasattr(target, "window_sql"):
    print(target.window_sql(manifest, window)[0][:1500])

# Right statistics keyed by Right column, re-keyed by Left column like ColCheck does
right_stats = target.window_statistics(manifest, window)
mapped = {left: right_stats[right] for left, right in manifest["col_map"].items()}

# Compare against the Left payloads of the same window
pointers = {row["column"]: row for _key, row in store.list_json("_INDEX/") if row["pair"] == PAIR and row["window"] == window["bucket"]}
rows = []
for column in manifest["columns"]:
    left_stats = store.get_json(pointers[column]["payload_key"])
    rows.append(compare_statistics(pair=PAIR, window=window["bucket"], column=column, left=left_stats, right=mapped[column]))
compared = pd.DataFrame(rows).set_index("column")
compared[["col_type", "verdict", "n_total_left", "n_total_right", "n_missing_diff", "n_unique_diff", "mean_match", "std_match", "min_match", "max_match", "hash_sum_match"]]

## 7. The drain scheduler

`DrainScheduler` decides what the Right job does next. A ready comparison (a Left unit whose Right window is already fetched) always outranks a speculative prefetch. Prefetches go first to windows that already have Left units, and at most `prefetch_depth` run ahead of them. This cell feeds it the real manifest and pointers in memory, so you can watch the order without touching any system.

In [ ]:
from dtrack_right.checks.col_check import DrainScheduler

# Seed it like ColCheck.poll does: the manifest (B-queue) and every published pointer
scheduler = DrainScheduler(prefetch_depth=2)
scheduler.add_manifest(copy.deepcopy(manifest))
for _key, pointer in store.list_json("_INDEX/"):
    if pointer["pair"] == PAIR:
        scheduler.add_unit(pointer)

# Pretend each prefetch succeeds with the window-0 statistics (in memory only) and record the order
order = []
for _ in range(manifest["expected_units"] + len(manifest["windows"]) + 2):
    task = scheduler.next_task()
    if task is None:
        break
    order.append((task.kind, task.window, task.payload.get("column", "")))
    if task.kind == "prefetch":
        scheduler.mark_right_ready(task.pair, task.window, mapped)
pd.DataFrame(order, columns=["kind", "window", "column"])

## 8. The column check drain (`ColCheck`)

`ColCheck.run` polls `manifest/` and `_INDEX/`, prefetches Right windows, compares units and writes `results/<unit>.json`. A unit that keeps failing is written to `dead/` after `max_attempts`. It returns once `manifest/_CLOSED` exists and every expected unit has a result or a dead letter. A second `ColCheck` on the same run skips everything already in `results/` and `dead/`, which is how a restarted job resumes without rework.

In [ ]:
from dtrack_right.checks.col_check import ColCheck

# Drain the run: every unit compared, results written to the scratch run
results = ColCheck(store, target_factory, prefetch_depth=3, max_attempts=3, idle_timeout=IDLE_TIMEOUT, poll_interval=2).run()
print(f"{len(results)} result(s) this pass:", pd.Series([r["verdict"] for r in results]).value_counts().to_dict())
print("dead units:", [row.get("error") for _key, row in store.list_json("dead/")] or "none")

# Resume: a fresh ColCheck finds nothing left to do
print("second pass:", len(ColCheck(store, target_factory, idle_timeout=IDLE_TIMEOUT, poll_interval=2).run()), "result(s)")
pd.DataFrame([row for _key, row in store.list_json("results/") if row["pair"] == PAIR]).set_index(["vintage", "column"])[["verdict", "n_total_diff", "n_unique_diff", "mean_match", "hash_sum_match", "min_match", "max_match"]]

## 9. The whole Right job loop (`dtrack_right.job.run`)

`job.run` is what `notebook.py` calls in Databricks. It resolves the run, writes `_STATUS.json` `right_waiting`, reconciles pending pairs, and drains with `ColCheck`, re-reconciling while idle. Then it builds the report (`summary.json`, CSVs, `index.html`, `report.xlsx`), publishes it to `report` (`history/<run_id>/` then `latest/pointer.json`) and marks the run `complete`. On this run everything is already done, so it reconciles and compares nothing new. That mirrors the "reruns without rework" test. It still builds and publishes the report into the scratch prefix.

In [ ]:
from dtrack_right.job import run as right_job

# The job's own loop, here, with this notebook's target; report to the scratch report store
outcome = right_job(
    RUN_URI, target_factory, report=REPORT_URI, out_dir=WORKDIR / "report", gate=GATE,
    poll_interval=2, idle_timeout=IDLE_TIMEOUT, wait_timeout=60,
)
summary = outcome["summary"]
print("new results:", outcome["results"], "| status:", summary["status"], "| findings:", summary["findings"])
print("units:", summary["units_completed"], "/", summary["units_expected"], "| errors:", summary["errors"] or "none")
print("published:", outcome["published"])
print("_STATUS.json:", store.get_json("_STATUS.json"), "| complete:", run_is_complete(store))
print("open:", (WORKDIR / "report" / "index.html").resolve())

# A second job run on the same run_id: nothing new, no report published without `report`
again = right_job(runs_root, target_factory, run_id=RUN_ID, out_dir=WORKDIR / "report_again", poll_interval=2, idle_timeout=IDLE_TIMEOUT, wait_timeout=60)
print("rerun: results", again["results"], "| units", again["summary"]["units_completed"], "| published", again["published"])

## 10. The real Databricks job (gated by `DRY_RUN`)

In production, `src/dtrack_right` is uploaded to the Workspace (`notebook.py` becomes a notebook, the rest plain files). The notebook then runs as a job with widgets `work`, `run_id`, `report`, `gate`, `idle_timeout`, `wait_timeout` and `aws_secret_scope`. That is how `tests/live/test_job_flow.py` does it. With `DRY_RUN = True`, the upload is a `dry_run` (planned paths only) and the submit is printed. With `DRY_RUN = False`, it uploads, submits a one-time run against a fresh scratch run and drives the Left side from here while the job runs. An `s3://` work location needs `AWS_SECRET_SCOPE` (or an instance profile on the compute).

In [ ]:
import threading
from datetime import timedelta

import dtrack_right
from dtrack_left.backends import databricks

# Upload the package (dry_run = planned remote paths only)
w = databricks.workspace(DBX_PROFILE)
folder = WORKSPACE_DIR or f"/Users/{w.current_user.me().user_name}/dtrack-nb13"
uploaded = databricks.upload_workspace(w, Path(dtrack_right.__file__).parent, f"{folder}/dtrack_right", overwrite=True, dry_run=DRY_RUN)
print({k: uploaded[k] for k in ("target", "dry_run", "planned", "uploaded", "failed")})
notebook_path = f"{folder}/dtrack_right/notebook"

# The widgets a scheduled job would pass; a fresh run id so the job has real work to do
JOB_RUN_ID = make_run_id(run_config) + "-job"
parameters = {
    "work": f"{NB_ROOT}/runs", "run_id": JOB_RUN_ID, "report": f"{REPORT_URI}/{JOB_RUN_ID}",
    "gate": GATE, "idle_timeout": "900", "wait_timeout": "1800", "aws_secret_scope": AWS_SECRET_SCOPE,
}
print("notebook  :", notebook_path)
print("parameters:", json.dumps(parameters, indent=2))

In [ ]:
from databricks.sdk.service import jobs

# Submit a one-time run (serverless unless DBX_COMPUTE is a cluster id) and play the Left side meanwhile
if DRY_RUN:
    print("DRY_RUN: nothing submitted. Set DRY_RUN = False to upload, submit and run the Left side against it.")
else:
    task = jobs.SubmitTask(
        task_key="dtrack_right",
        notebook_task=jobs.NotebookTask(notebook_path=notebook_path, base_parameters=parameters, source=jobs.Source.WORKSPACE),
        existing_cluster_id=None if DBX_COMPUTE == "serverless" else DBX_COMPUTE,
    )
    waiter = w.jobs.submit(run_name=f"dtrack-nb13-{JOB_RUN_ID}", tasks=[task])
    print("submitted run", waiter.run_id)
    finished = {}
    thread = threading.Thread(target=lambda: finished.update(run=waiter.result(timeout=timedelta(minutes=45))), daemon=True)
    thread.start()

    # The Left side, from this machine, into the run the job is watching
    job_store = open_store(f"{NB_ROOT}/runs/{JOB_RUN_ID}")
    left_row_check(run_config, job_store, run_id=JOB_RUN_ID, pair_names=[PAIR])
    job_manifests = wait_for_manifests(job_store, timeout=1800, poll_interval=5, pair_names=[PAIR])
    for item in job_manifests:
        publish_fixed_keys(job_store, item, WORKDIR / "job_keys")
    job_runtime = LeftRuntime(WORKDIR / "job_left", job_store)
    print("Left units:", ColCheckPipeline(job_store, pull_window=job_runtime.pull_window, compute_stats=job_runtime.compute_stats).run(job_manifests))
    thread.join(timeout=45 * 60)
    print("job result:", finished["run"].state.result_state, finished["run"].state.state_message or "")
    print("_STATUS.json:", job_store.get_json("_STATUS.json"))

# Read-only either way: the latest runs, newest first
pd.DataFrame(databricks.list_runs(w, limit=5)["runs"])

## Cleanup

This deletes this notebook's scratch objects under `NB_ROOT`: the run, the scratch report and the job run. When the run came from notebook 10, that is notebook 10's root. It also removes `WORKDIR`, and with `DRY_RUN = False` the uploaded Workspace folder. Nothing happens with `KEEP = True`.

In [ ]:
import shutil

from dtrack_left.backends.store import S3Store
from dtrack_left.checks import rollback

# Remove exactly what this notebook (and, for a handed-over run, notebook 10) created
if KEEP:
    print("kept", NB_ROOT)
else:
    scratch = S3Store(NB_ROOT)
    print("deleted", rollback.delete(scratch, scratch.list_keys("")), "object(s) under", NB_ROOT)
    shutil.rmtree(WORKDIR, ignore_errors=True)
    if not DRY_RUN:
        w.workspace.delete(folder, recursive=True)
        print("deleted Workspace folder", folder)

## What to check

- Section 2: preflight ok, with no `col_map` right names to fix. A case mismatch here explains numeric columns showing up as categorical later.
- Section 3: the Right daily counts line up with the Left ones. Every non-`match` day is a real finding, and the gate policy you use passes.
- Section 4: the verdict and CSV are written, the manifest is released, `row_check/_CLOSED` exists, and the second call reconciles 0.
- Section 6: `n_total_left == n_total_right` for every column in the window. Any `diff` row points to the statistic that differs.
- Section 7: prefetches come before compares, and every compare appears once its window is "ready".
- Section 8: one result per expected unit, no dead units, and the second pass returns 0.
- Section 9: `status` DONE (PARTIAL means dead or missing units), `latest/pointer.json` is published, `_STATUS.json` is `complete`, and the rerun adds 0 results.
- Section 10: the dry-run upload plans `.../dtrack_right/notebook`. A real run ends `SUCCESS`, with `_STATUS.json` `complete`.